# Retrieval v2 – Stage 0 / Stage 1 / Stage 2

This notebook is a thin control and inspection layer.

- **Stage 0** independently reproduces the frozen Baseline v1 retrieval behavior and must pass the strict historical per-query fixture.
- **Stage 1** evaluates L0–L4 lexical retrieval and freezes `L2_bm25_preserving` as the lexical default.
- **Stage 2** compares dense-only `E5-small`, `Qwen3-Embedding-0.6B`, and `BGE-M3` on the same four diagnostic ASR/text channels.
- Retrieval algorithms and metrics live in `src/`, not in notebook cells.
- Holdout20 remains closed.


## 1. Setup

Mount Google Drive, resolve the code root and artifact root, and verify that the Retrieval v2 files are present.

Normally no path edit is needed when the project is stored under `aic26/asr_model_comparison`. Use the override variables only when running from a different location.


In [1]:
from pathlib import Path
import glob
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount = False)
except Exception :
    pass


CODE_ROOT_OVERRIDE     = ""
ARTIFACT_ROOT_OVERRIDE = ""


def first_existing(candidates) :
    for candidate in candidates :
        path = Path(candidate)

        if (path.exists()) :
            return path.resolve()

    return None


def detect_artifact_root() :
    if (ARTIFACT_ROOT_OVERRIDE) :
        return Path(ARTIFACT_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_ARTIFACT_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        "/content/drive/MyDrive/aic26/asr_model_comparison",
    ]
    candidates.extend(
        glob.glob(
            "/content/drive/.shortcut-targets-by-id/*/aic26/asr_model_comparison"
        )
    )

    result = first_existing(candidates)

    if (result is None) :
        raise FileNotFoundError(
            "Could not detect the ASR artifact workspace. "
            "Set ARTIFACT_ROOT_OVERRIDE in this cell."
        )

    return result


def detect_code_root(artifact_root) :
    if (CODE_ROOT_OVERRIDE) :
        return Path(CODE_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_CODE_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        Path.cwd(),
        Path("/content/aic-2026-asr-retrieval"),
        artifact_root,
    ]

    for candidate in candidates :
        if ((candidate / "configs" / "retrieval_v2.json").exists()) :
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not detect the Retrieval v2 code root. "
        "Clone/copy the repository first or set CODE_ROOT_OVERRIDE."
    )


ARTIFACT_ROOT = detect_artifact_root()
CODE_ROOT     = detect_code_root(ARTIFACT_ROOT)

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["AIC_RETRIEVAL_CODE_ROOT"]     = str(CODE_ROOT)

RUNNER = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"
CONFIG = CODE_ROOT / "configs" / "retrieval_v2.json"

print("Code root    :", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Runner exists:", RUNNER.exists())
print("Config exists:", CONFIG.exists())

assert RUNNER.exists()
assert CONFIG.exists()

Mounted at /content/drive
Code root    : /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Runner exists: True
Config exists: True


## 2. Stage selection

Stage 0 and Stage 1 must already have passed their unchanged contracts before Stage 2 runs. Adding Stage 2 config does not invalidate earlier stages when their own contracts are unchanged.


In [2]:
STAGE = "stage02_dense"

assert STAGE in {"stage00_baseline", "stage01_lexical", "stage02_dense"}
print("Selected stage:", STAGE)

Selected stage: stage02_dense


## 3. Configuration and integrity preview

This cell shows the frozen benchmark identities, channels, and retrieval policy before execution.


In [3]:
config = json.loads(CONFIG.read_text(encoding = "utf-8"))

print("Retrieval ID :", config["retrieval_id"])
print("Frozen       :", config["retrieval_frozen"])
print("\nChannels:")
for channel_id, item in config["channels"].items() : print(f"  {channel_id:20s} model={item['model_id']:32s} view={item['view']}")

print("\nEvaluation policy:")
print(json.dumps(config["evaluation_policy"], ensure_ascii = False, indent = 2))

if (STAGE == "stage00_baseline") :
    print("\nStage 0 semantic model:"); print(json.dumps(config["stage00_baseline"]["semantic"], ensure_ascii = False, indent = 2))

elif (STAGE == "stage01_lexical") :
    print("\nStage 1 lexical methods:"); print(json.dumps(config["stage01_lexical"], ensure_ascii = False, indent = 2))

else :
    print("\nStage 2 dense backends:")
    for backend_id, item in config["stage02_dense"]["backends"].items() :
        print(f"  {backend_id:28s} {item['model_name']}@{item['revision'][:12]}")

    print("\nStage 2 score policy:");
    print(json.dumps(config["stage02_dense"]["score"], ensure_ascii = False, indent = 2))

print("\nStrict Stage 0 fixture:")
print(json.dumps(config["regression"], ensure_ascii = False, indent = 2))

Retrieval ID : aic2026_retrieval_v2_stage00_stage02
Frozen       : False

Channels:
  whisper_raw          model=whisper_large_v3                 view=raw
  whisper_processed    model=whisper_large_v3                 view=processed
  parakeet_raw         model=parakeet_ctc_0_6b_vietnamese     view=raw
  parakeet_processed   model=parakeet_ctc_0_6b_vietnamese     view=processed

Evaluation policy:
{
  "silver_radius_s": 60.0,
  "minimum_overlap_s": 30.0,
  "video_aggregation": "max_window",
  "tie_tolerance": 1e-12,
  "tie_policy": "worst_rank_within_tolerance",
  "story_scope": "correct_video_only"
}

Stage 2 dense backends:
  D0_e5_small                  intfloat/multilingual-e5-small@614241f622f5
  D1_e5_large_instruct         intfloat/multilingual-e5-large-instruct@274baa43b0e1
  D2_qwen3_embedding_0_6b      Qwen/Qwen3-Embedding-0.6B@97b0c614be4d
  D3_bge_m3_dense              BAAI/bge-m3@5617a9f61b02

Stage 2 score policy:
{
  "similarity": "cosine",
  "score_clip": null,
  "invali

## 4. Execute Retrieval v2

The runner streams live progress, validates all frozen identities, reuses valid embedding caches, evaluates rankings, writes reports, and exits non-zero on failure.


In [5]:
command = [sys.executable, "-u", str(RUNNER), "--stage", STAGE]
env     = {**os.environ, "PYTHONUNBUFFERED" : "1"}

print("Running:", " ".join(command), "\n")
process = subprocess.Popen(command, cwd = str(CODE_ROOT), env = env, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True, bufsize = 1)
for line in process.stdout : print(line, end = "", flush = True)
returncode = process.wait()
if (returncode != 0) : raise RuntimeError(f"Retrieval v2 failed with exit code {returncode}")

Running: /usr/bin/python3 -u /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py --stage stage02_dense 

AIC 2026 RETRIEVAL V2
Stage:         stage02_dense
Code root:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Reports:       /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage02_dense

[1/7] Loading configuration and benchmark manifests...
      done in 0.1s
[2/7] Validating query, corpus, and ASR artifact identity...
      whisper_large_v3/core10: loading 219 expected windows
      parakeet_ctc_0_6b_vietnamese/core10: loading 219 expected windows
      whisper_large_v3/extension40: loading 775 expected windows
      parakeet_ctc_0_6b_vietnamese/extension40: loa

## 5. Stage result

Load the machine-readable stage summary and validation report.


In [6]:
REPORT_ROOT = ARTIFACT_ROOT / config["paths"]["reports_root"] / STAGE

summary_path    = REPORT_ROOT / "stage_summary.json"
validation_path = REPORT_ROOT / "validation_summary.json"

summary    = json.loads(summary_path.read_text(encoding = "utf-8"))
validation = json.loads(validation_path.read_text(encoding = "utf-8"))

print("=" * 88)
print("STAGE RESULT")
print("=" * 88)
print("Stage :", summary["stage"])
print("Status:", "PASS" if summary["passed"] else "FAIL")
print("Reports:", REPORT_ROOT)

if ("strict_fixture" in summary) :
    print()
    print("Strict historical fixture:")
    print(json.dumps(summary["strict_fixture"], ensure_ascii = False, indent = 2))

print()
print("Validation:", "PASS" if validation["passed"] else "FAIL")


STAGE RESULT
Stage : stage02_dense
Status: PASS
Reports: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage02_dense

Strict historical fixture:
{
  "path": "/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/tests/fixtures/retrieval_v1_expected.json",
  "required": true,
  "exists": true,
  "checked": false,
  "file": {
    "passed": true,
    "required": true,
    "exists": true,
    "path": "/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/tests/fixtures/retrieval_v1_expected.json",
    "sha256": "78a8f97d4e0e349b16da023969b41e778c685a4ed5106836289802f3989d56b4",
    "expected_sha256": "78a8f97d4e0e349b16da023969b41e778c685a4ed5106836289802f3989d56b4",
    "fixture_id": "retrieval_v1_stage1_development_2026_08_09",
    "errors": []
  },
  "provenance": {
    "passed": true,
    "checks": [
      {
        "name

## 6. Aggregate metrics

Stage 0 reproduces Baseline v1, Stage 1 compares lexical methods, and Stage 2 compares the three dense backends independently across Whisper/Parakeet raw/processed.


In [7]:
metrics = pd.read_csv(REPORT_ROOT / "retrieval_metrics.csv")
metric_columns = ["query_set", "method_id", "model_id", "view", "query_count", "story_recall_at_1", "story_recall_at_3", "story_recall_at_5", "story_recall_at_10", "story_mrr", "video_recall_at_1", "video_recall_at_3", "video_recall_at_5", "video_recall_at_10", "video_recall_at_20", "video_mrr"]
display(metrics[[column for column in metric_columns if column in metrics.columns]].sort_values(["query_set", "method_id", "model_id", "view"]).reset_index(drop = True))

,query_set,method_id,model_id,view,query_count,story_recall_at_1,story_recall_at_3,story_recall_at_5,story_recall_at_10,story_mrr,video_recall_at_1,video_recall_at_3,video_recall_at_5,video_recall_at_10,video_recall_at_20,video_mrr
0,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,20,0.80,0.90,0.95,1.00,0.868750,0.50,0.60,0.70,0.85,0.90,0.588056
1,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,raw,20,0.75,0.95,0.95,1.00,0.848810,0.45,0.75,0.75,0.80,0.85,0.574416
2,development20,D0_e5_small,whisper_large_v3,processed,20,0.85,0.85,1.00,1.00,0.885000,0.50,0.80,0.80,0.85,0.90,0.628340
3,development20,D0_e5_small,whisper_large_v3,raw,20,0.75,0.80,1.00,1.00,0.817500,0.45,0.70,0.75,0.75,0.95,0.580902
4,development20,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,processed,20,0.75,0.90,1.00,1.00,0.825000,0.60,0.75,0.75,0.85,0.95,0.680781
5,development20,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,raw,20,0.70,0.95,0.95,1.00,0.833333,0.65,0.75,0.75,0.90,1.00,0.726452
6,development20,D1_e5_large_instruct,whisper_large_v3,processed,20,0.75,0.90,0.90,1.00,0.828810,0.65,0.80,0.85,0.95,0.95,0.753241
7,development20,D1_e5_large_instruct,whisper_large_v3,raw,20,0.85,0.90,0.90,1.00,0.877917,0.70,0.75,0.85,0.95,1.00,0.760084
8,development20,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,processed,20,0.75,0.90,1.00,1.00,0.839167,0.55,0.75,0.75,0.75,0.95,0.656992
9,development20,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,raw,20,0.70,0.95,0.95,1.00,0.814583,0.50,0.70,0.75,0.75,0.85,0.622983


## 7. Per-query comparison

Stage 1 compares lexical methods against L0. Stage 2 compares Qwen3 and BGE-M3 against dense control D0 E5-small and also reports the historical Baseline v1 system reference.


In [8]:
query_results = pd.read_csv(REPORT_ROOT / "query_results.csv")

if (STAGE == "stage01_lexical") :
    method_summary, query_comparison = pd.read_csv(REPORT_ROOT / "method_summary.csv"), pd.read_csv(REPORT_ROOT / "query_comparison.csv")
    print("Method-level paired summary:"); display(method_summary)
    print("\nPer-query changes vs L0:"); display(query_comparison.head(80))
elif (STAGE == "stage02_dense") :
    method_summary, query_comparison = pd.read_csv(REPORT_ROOT / "method_summary.csv"), pd.read_csv(REPORT_ROOT / "query_comparison.csv")
    backend_summary, asr_gap_summary = pd.read_csv(REPORT_ROOT / "backend_summary.csv"), pd.read_csv(REPORT_ROOT / "asr_gap_summary.csv")
    asr_gap_queries, system_reference = pd.read_csv(REPORT_ROOT / "asr_gap_queries.csv"), pd.read_csv(REPORT_ROOT / "system_reference_comparison.csv")
    print("Dense method paired summary vs D0 E5-small:"); display(method_summary)
    print("\nBackend/runtime summary:"); display(backend_summary)
    print("\nWhisper–Parakeet aggregate gap by dense backend:"); display(asr_gap_summary)
    print("\nWhisper–Parakeet paired query gaps:"); display(asr_gap_queries.head(120))
    print("\nDense-only performance vs historical Baseline v1:"); display(system_reference)
    print("\nPer-query changes vs D0:"); display(query_comparison.head(120))
else :
    print("Stage 0 query results:")
    columns = ["query_set", "model_id", "view", "query_id", "first_relevant_rank", "video_rank", "top_story_window_id", "top_video_id", "story_score_margin", "video_score_margin"]
    display(query_results[columns].head(120))

Dense method paired summary vs D0 E5-small:


,method_id,query_count,composite_video_rr_mean,composite_story_rr_mean,video_rr_delta_mean_vs_D0,video_rr_delta_median_vs_D0,story_rr_delta_mean_vs_D0,video_better_query_count_vs_D0,video_tie_query_count_vs_D0,video_worse_query_count_vs_D0,video_rr_delta_bootstrap_90_low,video_rr_delta_bootstrap_90_high,provisional_default
0,D0_e5_small,20,0.592929,0.855015,0.000000,0.000000,0.000000,0,20,0,0.000000,0.000000,NaN
1,D1_e5_large_instruct,20,0.730139,0.841265,0.137211,0.109944,-0.013750,12,6,2,0.056853,0.215074,NaN
2,D2_qwen3_embedding_0_6b,20,0.656930,0.831265,0.064001,0.000000,-0.023750,8,6,6,-0.047146,0.181665,NaN
3,D3_bge_m3_dense,20,0.568207,0.847202,-0.024721,0.000000,-0.007813,5,6,9,-0.088526,0.035890,NaN



Backend/runtime summary:


,backend_id,model_name,configured_revision,resolved_revision,device,embedding_dimension,normalization_policy,query_cache_hit,document_cache_hit_count,document_cache_miss_count,...,documents_encoded_this_run,documents_per_second_this_run,total_eligible_documents,model_load_runtime_s,last_effective_batch_size,online_latency_measured,cuda_available,device_name,peak_allocated_bytes,peak_reserved_bytes
0,D0_e5_small,intfloat/multilingual-e5-small,614241f622f53c4eeff9890bdc4f31cfecc418b3,614241f622f53c4eeff9890bdc4f31cfecc418b3,cuda,384,explicit_l2_float32_v1,False,0,4,...,3963,191.877320,3963,9.539581,1,True,True,Tesla T4,907521536,1247805440
1,D1_e5_large_instruct,intfloat/multilingual-e5-large-instruct,274baa43b0e13e37fafa6428dbc7938e62e5c439,274baa43b0e13e37fafa6428dbc7938e62e5c439,cuda,1024,explicit_l2_float32_v1,False,0,4,...,3963,103.632839,3963,11.963667,1,True,True,Tesla T4,1270973440,1428160512
2,D2_qwen3_embedding_0_6b,Qwen/Qwen3-Embedding-0.6B,97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3,97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3,cuda,1024,explicit_l2_float32_v1,False,0,4,...,3963,7.569872,3963,13.613551,1,True,True,Tesla T4,2686865408,3198156800
3,D3_bge_m3_dense,BAAI/bge-m3,5617a9f61b028005a4858fdac845db406aefb181,5617a9f61b028005a4858fdac845db406aefb181,cuda,1024,explicit_l2_float32_v1,False,0,4,...,3963,20.239799,3963,38.868664,1,True,True,Tesla T4,2560001024,2822766592



Whisper–Parakeet aggregate gap by dense backend:


...

,method_id,view,query_id,whisper_video_rank,parakeet_video_rank,video_rank_gap_parakeet_minus_whisper,whisper_story_rank,parakeet_story_rank,whisper_video_rr,parakeet_video_rr,video_rr_gap_whisper_minus_parakeet
0,D0_e5_small,processed,R1-13,3,5,2,1,1,0.333333,0.200000,0.133333
1,D0_e5_small,processed,R1-15,1,36,35,1,1,1.000000,0.027778,0.972222
2,D0_e5_small,processed,R1-17,1,1,0,1,1,1.000000,1.000000,0.000000
3,D0_e5_small,processed,R1-2,1,1,0,1,1,1.000000,1.000000,0.000000
4,D0_e5_small,processed,R1-22,1,1,0,1,2,1.000000,1.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
115,D2_qwen3_embedding_0_6b,raw,R3-10,1,1,0,1,1,1.000000,1.000000,0.000000
116,D2_qwen3_embedding_0_6b,raw,R3-15,1,1,0,1,1,1.000000,1.000000,0.000000
117,D2_qwen3_embedding_0_6b,raw,R3-2,28,13,-15,3,3,0.035714,0.076923,-0.041209
118,D2_qwen3_embedding_0_6b,raw,R3-20,1,5,4,1,2,1.000000,0.200000,0.800000



Dense-only performance vs historical Baseline v1:


,method_id,model_id,view,video_recall_at_1,baseline_v1_video_recall_at_1,video_recall_at_1_delta_vs_baseline_v1,video_mrr,baseline_v1_video_mrr,video_mrr_delta_vs_baseline_v1,story_mrr,baseline_v1_story_mrr,story_mrr_delta_vs_baseline_v1
0,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,0.50,0.35,0.15,0.588056,0.542095,0.045961,0.868750,0.761310,0.107440
1,D0_e5_small,parakeet_ctc_0_6b_vietnamese,raw,0.45,0.40,0.05,0.574416,0.569233,0.005183,0.848810,0.777045,0.071764
2,D0_e5_small,whisper_large_v3,processed,0.50,0.60,-0.10,0.628340,0.661035,-0.032695,0.885000,0.784545,0.100455
3,D0_e5_small,whisper_large_v3,raw,0.45,0.60,-0.15,0.580902,0.676134,-0.095231,0.817500,0.783056,0.034444
4,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,processed,0.60,0.35,0.25,0.680781,0.542095,0.138686,0.825000,0.761310,0.063690
5,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,raw,0.65,0.40,0.25,0.726452,0.569233,0.157219,0.833333,0.777045,0.056288
6,D1_e5_large_instruct,whisper_large_v3,processed,0.65,0.60,0.05,0.753241,0.661035,0.092206,0.828810,0.784545,0.044264
7,D1_e5_large_instruct,whisper_large_v3,raw,0.70,0.60,0.10,0.760084,0.676134,0.083950,0.877917,0.783056,0.094861
8,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,processed,0.55,0.35,0.20,0.656992,0.542095,0.114897,0.839167,0.761310,0.077857
9,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,raw,0.50,0.40,0.10,0.622983,0.569233,0.053750,0.814583,0.777045,0.037538



Per-query changes vs D0:


,query_set,model_id,view,query_id,reference_method,candidate_method,story_relation,video_relation,story_rr_delta,video_rr_delta,top_story_changed,top_video_changed
0,development20,whisper_large_v3,raw,R1-2,D0_e5_small,D1_e5_large_instruct,tie,tie,0.000,0.000000,False,False
1,development20,whisper_large_v3,raw,R1-9,D0_e5_small,D1_e5_large_instruct,worse,worse,-0.125,-0.083333,True,False
2,development20,whisper_large_v3,raw,R1-13,D0_e5_small,D1_e5_large_instruct,tie,better,0.000,0.750000,False,True
3,development20,whisper_large_v3,raw,R1-15,D0_e5_small,D1_e5_large_instruct,tie,tie,0.000,0.000000,False,False
4,development20,whisper_large_v3,raw,R1-17,D0_e5_small,D1_e5_large_instruct,tie,tie,0.000,0.000000,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
115,development20,whisper_large_v3,processed,R2-24,D0_e5_small,D2_qwen3_embedding_0_6b,tie,tie,0.000,0.000000,False,False
116,development20,whisper_large_v3,processed,R2-8,D0_e5_small,D2_qwen3_embedding_0_6b,worse,better,-0.050,0.046296,True,True
117,development20,whisper_large_v3,processed,R3-10,D0_e5_small,D2_qwen3_embedding_0_6b,tie,tie,0.000,0.000000,False,False
118,development20,whisper_large_v3,processed,R3-20,D0_e5_small,D2_qwen3_embedding_0_6b,tie,better,0.000,0.900000,True,True


## 8. Failure and evidence inspection

Stage 1 exposes lexical diagnostics. Stage 2 focuses on dense discrimination: score spread, top-k evidence concentration, correct-video support, and representative top windows/videos.


In [9]:
if (STAGE == "stage01_lexical") :
    diagnostics, evidence = pd.read_csv(REPORT_ROOT / "lexical_diagnostics.csv"), pd.read_csv(REPORT_ROOT / "top_evidence.csv")
    print("Lowest lexical coverage / zero-evidence cases:"); display(diagnostics.sort_values(["zero_evidence", "query_coverage"], ascending = [False, True]).head(40))
elif (STAGE == "stage02_dense") :
    diagnostics, evidence = pd.read_csv(REPORT_ROOT / "dense_diagnostics.csv"), pd.read_csv(REPORT_ROOT / "top_evidence.csv")
    print("Worst dense video margins:"); display(diagnostics.sort_values(["video_score_margin", "video_rank"], ascending = [True, False]).head(40))
else :
    regression = json.loads((REPORT_ROOT / "regression.json").read_text(encoding = "utf-8"))
    print("Stage 0 regression summary:"); print(json.dumps(summary["regression"], ensure_ascii = False, indent = 2))
    evidence = None

if (STAGE in {"stage01_lexical", "stage02_dense"}) :
    QUERY_ID = "R2-20" if "R2-20" in set(evidence["query_id"]) else evidence["query_id"].iloc[0]
    print("\nEvidence for query:", QUERY_ID)
    display(evidence[evidence["query_id"] == QUERY_ID].sort_values(["method_id", "model_id", "view", "entity_type", "rank"]).head(160))

Worst dense video margins:


,query_set,method_id,model_id,view,channel_id,query_id,eligible_window_count,score_mean,score_std,top_score,...,top5_score_mean,correct_video_windows_in_top5,top20_score_mean,correct_video_windows_in_top20,first_relevant_rank,video_rank,correct_video_score,best_wrong_video_score,video_score_margin,story_score_margin
210,development20,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,raw,parakeet_raw,R2-1,994,0.246340,0.068741,0.491849,...,0.466770,0,0.434005,0,1,21,0.376717,0.491849,-0.115132,0.038723
171,development20,D2_qwen3_embedding_0_6b,whisper_large_v3,raw,whisper_raw,R2-3,994,0.251961,0.071926,0.494875,...,0.463059,0,0.435434,0,1,20,0.387844,0.494875,-0.107031,0.016136
250,development20,D3_bge_m3_dense,whisper_large_v3,raw,whisper_raw,R2-1,994,0.361441,0.046390,0.555265,...,0.504101,0,0.480574,0,3,20,0.449560,0.555265,-0.105705,-0.030283
230,development20,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,processed,parakeet_processed,R2-1,994,0.232828,0.070089,0.481614,...,0.457160,0,0.421615,0,1,20,0.378955,0.481614,-0.102659,0.015765
236,development20,D2_qwen3_embedding_0_6b,parakeet_ctc_0_6b_vietnamese,processed,parakeet_processed,R2-8,994,0.257404,0.073530,0.477996,...,0.454234,0,0.433492,0,5,31,0.376433,0.477996,-0.101563,-0.082188
270,development20,D3_bge_m3_dense,whisper_large_v3,processed,whisper_processed,R2-1,981,0.365279,0.047548,0.550678,...,0.513913,0,0.489789,0,2,20,0.452551,0.550678,-0.098126,-0.008884
276,development20,D3_bge_m3_dense,whisper_large_v3,processed,whisper_processed,R2-8,981,0.394169,0.050606,0.557128,...,0.528591,0,0.508712,0,1,38,0.463389,0.557128,-0.093738,0.056631
168,development20,D2_qwen3_embedding_0_6b,whisper_large_v3,raw,whisper_raw,R3-2,994,0.250744,0.064351,0.444450,...,0.422785,0,0.402409,0,3,28,0.356282,0.444450,-0.088169,-0.040807
174,development20,D2_qwen3_embedding_0_6b,whisper_large_v3,raw,whisper_raw,R2-20,994,0.279140,0.066182,0.465958,...,0.454737,0,0.427784,0,1,37,0.377972,0.465958,-0.087986,0.030851
268,development20,D3_bge_m3_dense,whisper_large_v3,processed,whisper_processed,R3-2,981,0.378817,0.045536,0.557839,...,0.541551,0,0.497851,0,11,13,0.472418,0.557839,-0.085421,-0.037004



Evidence for query: R2-20


,query_set,method_id,model_id,view,query_id,entity_type,rank,entity_id,video_id,start_s,end_s,score,correct,relevant,eligible,zero_reason,silver_overlap_s
2240,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,R2-20,video,1,K08_V019,K08_V019,NaN,NaN,0.855505,False,False,True,NaN,NaN
2241,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,R2-20,video,2,K05_V013,K05_V013,NaN,NaN,0.850160,False,False,True,NaN,NaN
2242,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,R2-20,video,3,K12_V007,K12_V007,NaN,NaN,0.849163,False,False,True,NaN,NaN
2243,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,R2-20,video,4,K02_V012,K02_V012,NaN,NaN,0.848509,False,False,True,NaN,NaN
2244,development20,D0_e5_small,parakeet_ctc_0_6b_vietnamese,processed,R2-20,video,5,K03_V019,K03_V019,NaN,NaN,0.847552,False,False,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4045,development20,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,raw,R2-20,video,6,K06_V022,K06_V022,NaN,NaN,0.865272,False,False,True,NaN,NaN
4046,development20,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,raw,R2-20,video,7,L21_V029,L21_V029,NaN,NaN,0.864594,False,False,True,NaN,NaN
4047,development20,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,raw,R2-20,video,8,K05_V018,K05_V018,NaN,NaN,0.863416,False,False,True,NaN,NaN
4048,development20,D1_e5_large_instruct,parakeet_ctc_0_6b_vietnamese,raw,R2-20,video,9,L27_V010,L27_V010,NaN,NaN,0.862566,False,False,True,NaN,NaN


## 9. Research decision

Record the decision only after reviewing aggregate metrics, paired ranks, backend/runtime behavior, ASR gaps, and failure evidence. Stage 2 must select at most one dense default and one close alternative; no ASR or text-view fusion is introduced.


In [10]:
RETRIEVAL_DECISION = {
    "stage" : STAGE,
    "lexical_default" : "L2_bm25_preserving",
    "provisional_dense_default" : None,
    "retained_dense_alternative" : None,
    "main_observation" : None,
    "failure_observation" : None,
    "asr_gap_observation" : None,
    "notes" : ["Stage 1 lexical default is frozen as L2_bm25_preserving.", "Stage 2 compares dense representations only; max-window aggregation remains frozen.", "Whisper and Parakeet remain independent diagnostic channels; no ASR fusion is planned.", "Raw and processed remain diagnostic views; no text-view fusion is assumed.", "Holdout20 remains closed."],
}
print(json.dumps(RETRIEVAL_DECISION, ensure_ascii = False, indent = 2))

{
  "stage": "stage02_dense",
  "lexical_default": "L2_bm25_preserving",
  "provisional_dense_default": null,
  "retained_dense_alternative": null,
  "main_observation": null,
  "failure_observation": null,
  "asr_gap_observation": null,
  "notes": [
    "Stage 1 lexical default is frozen as L2_bm25_preserving.",
    "Stage 2 compares dense representations only; max-window aggregation remains frozen.",
    "Whisper and Parakeet remain independent diagnostic channels; no ASR fusion is planned.",
    "Raw and processed remain diagnostic views; no text-view fusion is assumed.",
    "Holdout20 remains closed."
  ]
}
